# C-Index STEP=2 Benchmark Filters

Contribution 확인용 benchmark 전용 노트북.

이 노트북은 **모델 학습, XAI 추출, C-index 재계산을 하지 않는다.**
이미 저장된 `finalDf_cache.pkl`만 불러와서 다음 두 baseline과 C-index filter를 비교한다.

## Benchmarks

1. **Prediction Confidence Filter**
   - 질문: C-index가 아니라 단순히 예측 확률이 높은 거래만 골라도 같은 효과가 나는가?
   - 방식: `y_hat == 1`인 거래 중 `prob` 상위 구간만 거래.

2. **Random Count-Matched Filter**
   - 질문: C-index가 좋아서가 아니라 단순히 거래 수를 줄여서 좋아진 것인가?
   - 방식: C-index best와 동일한 거래 수만큼 No Filter 거래에서 무작위 선택. 1000회 반복.

## Interpretation

- C-index가 Confidence Filter보다 좋으면, 설명 합의도가 예측확률 이상의 정보를 제공할 가능성이 있다.
- C-index가 Random Count-Matched보다 좋으면, 단순 turnover 감소 이상의 선택 효과가 있을 가능성이 있다.
- 반대로 benchmark와 비슷하거나 밀리면, C-index의 contribution은 성과 개선보다 reliability diagnostic으로 낮춰 해석한다.

In [ ]:
# Mount Drive in Colab; local audit runs use CINDEX_PROJECT_DIR instead.
import os
if not os.environ.get('CINDEX_PROJECT_DIR'):
    from google.colab import drive
    drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
Q_LIST = [0.2, 0.3, 0.4, 0.5]
COST_BPS_LIST = [0, 5, 10]
N_RANDOM = 1000
MIN_TRADES_POOLED = 15
TRADING_DAYS_PER_YEAR = 252
SHARPE_EPSILON = 1e-6

DRIVE_PROJECT_DIR = Path('/content/drive/MyDrive/c-index')
DEFAULT_PROJECT_DIR = DRIVE_PROJECT_DIR if DRIVE_PROJECT_DIR.exists() else Path('/content/c-index')
PROJECT_DIR = Path(os.environ.get('CINDEX_PROJECT_DIR', DEFAULT_PROJECT_DIR))
ARTIFACT_DIR = PROJECT_DIR / 'artifacts'
RESULTS_DIR = PROJECT_DIR / 'results'
OUTPUT_DIR = RESULTS_DIR / 'figures'
TABLES_DIR = RESULTS_DIR / 'tables'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)

FINAL_DF_CACHE = ARTIFACT_DIR / 'finalDf_cache.pkl'

random.seed(SEED)
np.random.seed(SEED)
rng = np.random.default_rng(SEED)

print('ARTIFACT_DIR:', ARTIFACT_DIR)
print('OUTPUT_DIR:', OUTPUT_DIR)
print('N_RANDOM:', N_RANDOM)

In [ ]:
# Load finalDf only. No model training, no XAI recomputation.
if not FINAL_DF_CACHE.exists():
    raise FileNotFoundError(f'finalDf cache not found: {FINAL_DF_CACHE}')

finalDf = pd.read_pickle(FINAL_DF_CACHE)
finalDf['date'] = pd.to_datetime(finalDf['date'])
finalDf = finalDf.sort_values(['model', 'date', 'asset']).reset_index(drop=True)

required_cols = {'model', 'asset', 'date', 'ret_1d_next', 'prob', 'y_hat'}
missing = required_cols - set(finalDf.columns)
if missing:
    raise KeyError(f'Missing required columns: {sorted(missing)}')

c_index_cols = list(finalDf.filter(like='C_', axis=1).columns)
if not c_index_cols:
    raise ValueError('No C-index columns found in finalDf.')

print('[cache loaded]', FINAL_DF_CACHE)
print('shape:', finalDf.shape)
print('models:', sorted(finalDf['model'].unique()))
print('assets:', sorted(finalDf['asset'].unique()))
print('date range:', finalDf['date'].min(), 'to', finalDf['date'].max())
print('C-index columns:', c_index_cols)

display(finalDf.head())

In [ ]:
def _prepare_trade_frame(trades, return_col='ret_1d_next', date_col='date'):
    out = trades.copy()
    if return_col not in out.columns:
        raise KeyError(f'{return_col} is required.')
    out = out.rename(columns={return_col: 'ret'})
    out['date'] = pd.to_datetime(out[date_col]) if date_col in out.columns else pd.NaT
    return out[['date', 'ret']].dropna(subset=['ret'])


def _max_drawdown(return_series):
    returns = pd.Series(return_series).dropna()
    if returns.empty:
        return np.nan
    equity = returns.cumsum()
    return float((equity - equity.cummax()).min() * 100)


def compute_metrics(trades, min_trades=MIN_TRADES_POOLED, calendar_dates=None):
    trade_df = _prepare_trade_frame(trades)
    if trade_df['date'].notna().any():
        # Stable sort makes same-day pooled trades deterministic across reruns.
        trade_df = trade_df.sort_values('date', kind='mergesort').reset_index(drop=True)
    rets = trade_df['ret'].dropna()
    n = len(rets)

    if calendar_dates is not None:
        eval_days = len(pd.to_datetime(pd.Series(calendar_dates).dropna().unique()))
    elif trade_df['date'].notna().any():
        eval_days = len(pd.to_datetime(trade_df['date'].dropna().unique()))
    else:
        eval_days = np.nan

    annual_trades = n / (eval_days / TRADING_DAYS_PER_YEAR) if pd.notna(eval_days) and eval_days > 0 else np.nan

    if n == 0:
        return {
            'Trade Count': 0,
            'Avg Return': np.nan,
            'Win Rate': np.nan,
            'Sharpe': np.nan,
            'Sharpe_annualized': np.nan,
            'Annual Trades': float(annual_trades) if pd.notna(annual_trades) else np.nan,
            'Eval Days': int(eval_days) if pd.notna(eval_days) else np.nan,
            'Max Drawdown': np.nan,
            'Note': ''
        }

    avg_ret = float(rets.mean() * 100)
    win_rate = float((rets > 0).mean() * 100)
    mdd = _max_drawdown(rets)

    if n < min_trades:
        sharpe = np.nan
        sharpe_ann = np.nan
        note = f'N={n}<{min_trades}'
    else:
        std = rets.std()
        if std <= SHARPE_EPSILON:
            sharpe = np.nan
            sharpe_ann = np.nan
            note = 'std≈0'
        else:
            sharpe = float(rets.mean() / std)
            sharpe_ann = float(sharpe * np.sqrt(annual_trades)) if pd.notna(annual_trades) else np.nan
            note = ''

    return {
        'Trade Count': int(n),
        'Avg Return': avg_ret,
        'Win Rate': win_rate,
        'Sharpe': sharpe,
        'Sharpe_annualized': sharpe_ann,
        'Annual Trades': float(annual_trades) if pd.notna(annual_trades) else np.nan,
        'Eval Days': int(eval_days) if pd.notna(eval_days) else np.nan,
        'Max Drawdown': float(mdd),
        'Note': note
    }


def apply_cost(trades, cost_bps=0):
    out = trades.copy()
    out['ret_1d_next'] = out['ret_1d_next'] - cost_bps / 10000.0
    return out


def evaluate_trades(trades, sub, cost_bps, method, model_name, q=np.nan, c_col=np.nan, threshold=np.nan):
    net_trades = apply_cost(trades[['date', 'ret_1d_next']].copy(), cost_bps=cost_bps)
    m = compute_metrics(net_trades, min_trades=MIN_TRADES_POOLED, calendar_dates=sub['date'].unique())
    m.update({
        'Model': model_name,
        'Method': method,
        'Cost (bp)': cost_bps,
        'C-Index Type': c_col,
        'Quantile': q,
        'Threshold Value': threshold,
    })
    return m

In [ ]:
def evaluate_no_filter(sub, cost_bps, model_name):
    trades = sub.loc[sub['y_hat'] == 1, ['date', 'ret_1d_next']].copy()
    return evaluate_trades(trades, sub, cost_bps, 'No Filter', model_name)


def evaluate_cindex_candidate(sub, c_col, q, cost_bps, model_name):
    threshold = sub[c_col].quantile(q)
    is_trade = (sub['y_hat'] == 1) & (sub[c_col] >= threshold)
    trades = sub.loc[is_trade, ['date', 'ret_1d_next']].copy()
    return evaluate_trades(trades, sub, cost_bps, 'C-index Filter', model_name, q=q, c_col=c_col, threshold=threshold)


def evaluate_confidence_candidate(sub, q, cost_bps, model_name):
    base = sub[sub['y_hat'] == 1].copy()
    if base.empty:
        threshold = np.nan
        trades = base[['date', 'ret_1d_next']].copy()
    else:
        threshold = base['prob'].quantile(q)
        trades = base.loc[base['prob'] >= threshold, ['date', 'ret_1d_next']].copy()
    return evaluate_trades(trades, sub, cost_bps, 'Prediction Confidence Filter', model_name, q=q, c_col='prob', threshold=threshold)


def random_count_matched_summary(sub, target_n, cost_bps, model_name, reference_method, n_random=N_RANDOM, seed=SEED):
    base = sub.loc[sub['y_hat'] == 1, ['date', 'ret_1d_next']].copy().reset_index(drop=True)
    base_n = len(base)
    target_n = int(target_n)
    model_offset = {'gbm': 1000, 'mlp': 2000, 'rf': 3000}.get(str(model_name), 4000)
    local_seed = int(seed + model_offset + cost_bps * 100 + target_n)
    local_rng = np.random.default_rng(local_seed)

    rows = []
    if base_n == 0 or target_n <= 0 or target_n > base_n:
        return pd.DataFrame([{
            'Model': model_name,
            'Method': 'Random Count-Matched Filter',
            'Cost (bp)': cost_bps,
            'Reference Method': reference_method,
            'Target Trade Count': target_n,
            'Random Iterations': n_random,
            'Random Sharpe Mean': np.nan,
            'Random Sharpe Median': np.nan,
            'Random Sharpe 5%': np.nan,
            'Random Sharpe 95%': np.nan,
            'Random Avg Return Mean': np.nan,
            'Random MDD Mean': np.nan,
            'Note': 'invalid target_n'
        }])

    for i in range(n_random):
        idx = local_rng.choice(base_n, size=target_n, replace=False)
        trades = base.iloc[idx].copy()
        trades = trades.sort_values('date', kind='mergesort')
        m = evaluate_trades(trades, sub, cost_bps, 'Random Count-Matched Filter', model_name)
        rows.append(m)

    dist = pd.DataFrame(rows)
    return pd.DataFrame([{
        'Model': model_name,
        'Method': 'Random Count-Matched Filter',
        'Cost (bp)': cost_bps,
        'Reference Method': reference_method,
        'Target Trade Count': target_n,
        'Random Iterations': n_random,
        'Random Sharpe Mean': dist['Sharpe'].mean(),
        'Random Sharpe Median': dist['Sharpe'].median(),
        'Random Sharpe 5%': dist['Sharpe'].quantile(0.05),
        'Random Sharpe 95%': dist['Sharpe'].quantile(0.95),
        'Random Avg Return Mean': dist['Avg Return'].mean(),
        'Random Avg Return Median': dist['Avg Return'].median(),
        'Random MDD Mean': dist['Max Drawdown'].mean(),
        'Random MDD Median': dist['Max Drawdown'].median(),
        'Note': ''
    }])

## Benchmark Evaluation Plan

각 model-cost pair에 대해 다음을 계산한다.

1. `No Filter`
2. `C-index Filter`: 기존과 동일하게 6개 C-index × 4개 q 후보 중 Sharpe 기준 best
3. `Prediction Confidence Filter`: q 후보 중 Sharpe 기준 best
4. `Random Count-Matched Filter`: C-index best와 동일한 거래 수를 랜덤 선택한 1000회 분포

주의: C-index와 Confidence 모두 q 후보 중 best를 고르므로 이 표는 contribution 탐색용 benchmark다. 통계적 유의성 확정 근거가 아니라, "C-index가 단순 confidence/random baseline 대비 어떤 위치에 있는지" 확인하는 용도다.

In [ ]:
benchmark_rows = []
random_rows = []

for model_name, sub in finalDf.groupby('model'):
    sub = sub.copy().sort_values(['date', 'asset']).reset_index(drop=True)

    for cost_bps in COST_BPS_LIST:
        # 1) No Filter
        no_filter = evaluate_no_filter(sub, cost_bps, model_name)
        baseline_trade_count = int(no_filter['Trade Count'])
        benchmark_rows.append(no_filter)

        # 2) C-index candidates: choose best by per-trade Sharpe, same as presentation/transaction-cost result style.
        c_rows = []
        for c_col in c_index_cols:
            for q in Q_LIST:
                c_rows.append(evaluate_cindex_candidate(sub, c_col, q, cost_bps, model_name))
        c_grid = pd.DataFrame(c_rows)
        c_valid = c_grid[
            (c_grid['Trade Count'] >= MIN_TRADES_POOLED) &
            (c_grid['Trade Count'] < baseline_trade_count) &
            np.isfinite(c_grid['Sharpe'])
        ].copy()
        if c_valid.empty:
            raise ValueError(f'No active C-index candidate: model={model_name}, cost={cost_bps}')
        else:
            c_best = c_valid.sort_values('Sharpe', ascending=False).head(1).iloc[0].to_dict()
        c_best['Method'] = 'C-index Filter (Best Sharpe)'
        benchmark_rows.append(c_best)

        # 3) Prediction confidence candidates: choose best by per-trade Sharpe.
        conf_rows = []
        for q in Q_LIST:
            conf_rows.append(evaluate_confidence_candidate(sub, q, cost_bps, model_name))
        conf_grid = pd.DataFrame(conf_rows)
        conf_valid = conf_grid[
            (conf_grid['Trade Count'] >= MIN_TRADES_POOLED) &
            (conf_grid['Trade Count'] < baseline_trade_count) &
            np.isfinite(conf_grid['Sharpe'])
        ].copy()
        if conf_valid.empty:
            raise ValueError(f'No active confidence candidate: model={model_name}, cost={cost_bps}')
        else:
            conf_best = conf_valid.sort_values('Sharpe', ascending=False).head(1).iloc[0].to_dict()
        conf_best['Method'] = 'Prediction Confidence Filter (Best Sharpe)'
        benchmark_rows.append(conf_best)

        # 4) Random count-matched to C-index best trade count.
        random_rows.append(
            random_count_matched_summary(
                sub=sub,
                target_n=c_best['Trade Count'],
                cost_bps=cost_bps,
                model_name=model_name,
                reference_method='C-index Filter (Best Sharpe)'
            )
        )
        print(f'[benchmark done] model={model_name} cost={cost_bps} '              f'cindex={c_best["C-Index Type"]} q={c_best["Quantile"]} '              f'trades={int(c_best["Trade Count"])}')

benchmark_summary = pd.DataFrame(benchmark_rows)
random_count_matched_summary_table = pd.concat(random_rows, ignore_index=True)

cols = ['Model', 'Cost (bp)', 'Method', 'Trade Count', 'Avg Return', 'Win Rate', 'Sharpe', 'Sharpe_annualized', 'Annual Trades', 'Max Drawdown', 'C-Index Type', 'Quantile', 'Threshold Value', 'Note']
benchmark_summary = benchmark_summary[[c for c in cols if c in benchmark_summary.columns]].sort_values(['Cost (bp)', 'Model', 'Method']).reset_index(drop=True)

print('benchmark_summary:', benchmark_summary.shape)
print('random_count_matched_summary_table:', random_count_matched_summary_table.shape)

display(benchmark_summary.round(6))
display(random_count_matched_summary_table.round(6))

In [ ]:
# Compare C-index best against confidence best and random count-matched benchmark.
wide = benchmark_summary.pivot_table(
    index=['Model', 'Cost (bp)'],
    columns='Method',
    values=['Trade Count', 'Avg Return', 'Win Rate', 'Sharpe', 'Max Drawdown'],
    aggfunc='first'
)
wide.columns = [f'{metric} | {method}' for metric, method in wide.columns]
wide = wide.reset_index()

comparison_rows = []
for _, row in wide.iterrows():
    model_name = row['Model']
    cost_bps = row['Cost (bp)']
    rand = random_count_matched_summary_table[
        (random_count_matched_summary_table['Model'] == model_name) &
        (random_count_matched_summary_table['Cost (bp)'] == cost_bps)
    ].iloc[0]

    c_sharpe = row.get('Sharpe | C-index Filter (Best Sharpe)', np.nan)
    conf_sharpe = row.get('Sharpe | Prediction Confidence Filter (Best Sharpe)', np.nan)
    no_sharpe = row.get('Sharpe | No Filter', np.nan)
    c_avg = row.get('Avg Return | C-index Filter (Best Sharpe)', np.nan)
    conf_avg = row.get('Avg Return | Prediction Confidence Filter (Best Sharpe)', np.nan)
    c_mdd = row.get('Max Drawdown | C-index Filter (Best Sharpe)', np.nan)
    conf_mdd = row.get('Max Drawdown | Prediction Confidence Filter (Best Sharpe)', np.nan)

    comparison_rows.append({
        'Model': model_name,
        'Cost (bp)': cost_bps,
        'C-index Sharpe': c_sharpe,
        'Confidence Sharpe': conf_sharpe,
        'No Filter Sharpe': no_sharpe,
        'C-index - Confidence Sharpe': c_sharpe - conf_sharpe,
        'C-index - No Filter Sharpe': c_sharpe - no_sharpe,
        'Random Sharpe Mean': rand['Random Sharpe Mean'],
        'Random Sharpe 95%': rand['Random Sharpe 95%'],
        'C-index > Random 95%': bool(c_sharpe > rand['Random Sharpe 95%']) if pd.notna(c_sharpe) and pd.notna(rand['Random Sharpe 95%']) else False,
        'C-index Avg Return': c_avg,
        'Confidence Avg Return': conf_avg,
        'C-index MDD': c_mdd,
        'Confidence MDD': conf_mdd,
    })

benchmark_comparison = pd.DataFrame(comparison_rows)

display(benchmark_comparison.round(6))

In [ ]:
# Save outputs flat under artifacts.
outputs = {
    'benchmark_filter_summary': benchmark_summary,
    'benchmark_random_count_matched_summary': random_count_matched_summary_table,
    'benchmark_filter_comparison': benchmark_comparison,
}

for name, df_out in outputs.items():
    pkl_path = ARTIFACT_DIR / f'{name}.pkl'
    csv_path = TABLES_DIR / f'{name}.csv'
    df_out.to_pickle(pkl_path)
    df_out.to_csv(csv_path, index=False)
    print('[saved]', pkl_path)
    print('[saved]', csv_path)

In [ ]:
def save_df_pretty(df, filename, dpi=450, title=None, include_index=False):
    df_show = df.copy()
    if include_index:
        df_show = df_show.reset_index()

    nrows, ncols = df_show.shape
    fig_w = max(10, ncols * 1.65)
    fig_h = max(2.5, (nrows + 1) * 0.52)
    fig, ax = plt.subplots(figsize=(fig_w, fig_h))
    ax.axis('off')
    if title:
        ax.set_title(title, fontsize=13, pad=14)
    tbl = ax.table(cellText=df_show.values, colLabels=df_show.columns, loc='center', cellLoc='center')
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(9)
    tbl.scale(1.08, 1.55)
    for (r, c), cell in tbl.get_celld().items():
        cell.set_linewidth(1.0)
        if r == 0:
            cell.set_text_props(weight='bold')
            cell.set_height(cell.get_height() * 1.12)
    plt.tight_layout(pad=1.5)
    fig.savefig(OUTPUT_DIR / filename, dpi=dpi, bbox_inches='tight', facecolor='white')
    plt.close(fig)

save_df_pretty(benchmark_summary.round(6), 'benchmark_filter_summary.png', title='Benchmark Filter Summary')
save_df_pretty(random_count_matched_summary_table.round(6), 'benchmark_random_count_matched_summary.png', title='Random Count-Matched Benchmark')
save_df_pretty(benchmark_comparison.round(6), 'benchmark_filter_comparison.png', title='C-index vs Confidence vs Random Benchmark')

print('saved benchmark figures to', OUTPUT_DIR)

## How to Read the Result

### 1. Prediction Confidence Filter와 비교

- C-index Sharpe > Confidence Sharpe이면, 설명 합의도가 단순 예측확률 이상의 정보를 줄 가능성이 있다.
- Confidence Sharpe가 C-index와 비슷하거나 더 높으면, C-index의 성과 개선 contribution은 약해지고 reliability diagnostic 프레이밍을 유지하는 것이 안전하다.

### 2. Random Count-Matched Filter와 비교

- C-index Sharpe가 random 95%보다 높으면, 단순 turnover 감소 이상의 선택 효과가 있을 가능성이 있다.
- C-index Sharpe가 random 평균 근처이면, 성과 개선은 거래 수 감소 효과일 가능성이 크다.

### 3. PPT 반영 방식

결과가 좋을 경우:

```text
C-index filter는 단순 confidence filter 및 동일 거래 수 random filter 대비 높은 risk-adjusted performance를 보여, 설명 합의도가 거래 선택에 추가 정보를 제공할 가능성을 확인했다.
```

결과가 애매할 경우:

```text
Benchmark 비교 결과, C-index filter의 성과 개선은 일부 조건에서 관측되었으나 confidence/random baseline 대비 일관된 우위는 아직 확인되지 않았다. 따라서 본 프로젝트는 C-index를 성과 보장 도구가 아니라 모델 판단의 reliability diagnostic으로 제안한다.
```